# AQI-Intel: Hyperlocal Predictive AQI Forecasting

Training notebook for the AQI prediction model.  
Designed to run top-to-bottom on Google Colab.

**Sections:**
1. Setup
2. Load data
3. EDA
4. Feature engineering
5. Train/test split
6. Persistence baseline
7. Model training
8. Evaluation
9. Export
10. Future improvements

---
## 1. Setup

In [ ]:
# Install dependencies (Colab has most of these, but pin versions)
!pip install -q xgboost lightgbm scikit-learn joblib matplotlib seaborn

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

# Plotting defaults
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'XGBoost: {xgb.__version__}')

In [ ]:
# Mount Google Drive (if running on Colab and data is on Drive)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    print('Google Drive mounted at /content/drive')
except ImportError:
    DRIVE_MOUNTED = False
    print('Not on Colab — skipping Drive mount')

In [ ]:
# === CONFIGURE THIS ===
# Path to the Kaggle 'city_day.csv' (or 'station_day.csv')
# On Colab with Drive: '/content/drive/MyDrive/aqi-intel/city_day.csv'
# Local: '../data/raw/city_day.csv'

CSV_PATH = '../data/raw/city_day.csv'

# If on Drive, override:
if DRIVE_MOUNTED and not os.path.exists(CSV_PATH):
    CSV_PATH = '/content/drive/MyDrive/aqi-intel/city_day.csv'

print(f'Will load data from: {CSV_PATH}')

---
## 2. Load Data

In [ ]:
df_raw = pd.read_csv(CSV_PATH, parse_dates=['Date'])
print(f'Shape: {df_raw.shape}')
print(f'Date range: {df_raw["Date"].min()} to {df_raw["Date"].max()}')
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
# Missingness check
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
print('Missing values (%):')
print(missing_pct.to_string())

# Visualize missingness
fig, ax = plt.subplots(figsize=(12, 5))
missing_pct.plot.bar(ax=ax, color='#e74c3c', alpha=0.8)
ax.set_title('Missing Values by Column (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('% Missing')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 3. Exploratory Data Analysis

In [ ]:
# AQI distribution per city (top 10 cities by row count)
top_cities = df_raw['City'].value_counts().head(10).index.tolist()
df_top = df_raw[df_raw['City'].isin(top_cities)].copy()

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_top, x='City', y='AQI', palette='viridis', ax=ax)
ax.set_title('AQI Distribution — Top 10 Cities', fontsize=14, fontweight='bold')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Time series plot for Delhi and Mumbai
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, city in zip(axes, ['Delhi', 'Mumbai']):
    city_df = df_raw[df_raw['City'] == city].sort_values('Date')
    ax.plot(city_df['Date'], city_df['AQI'], alpha=0.7, linewidth=0.8)
    ax.set_title(f'{city} — AQI Over Time', fontsize=13, fontweight='bold')
    ax.set_ylabel('AQI')
    ax.axhline(y=200, color='red', linestyle='--', alpha=0.5, label='Poor threshold')
    ax.legend()

plt.xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for pollutant columns + AQI
pollutant_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'AQI']
pollutant_cols = [c for c in pollutant_cols if c in df_raw.columns]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_raw[pollutant_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, ax=ax)
ax.set_title('Pollutant Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Feature Engineering

All transforms are plain pandas — no sklearn Pipeline/ColumnTransformer.

Features:
- **Lag features**: AQI at t-1, t-24, t-168 (1 day ago, 1 week ago; dataset is daily so t-1 = yesterday)
- **Rolling stats**: mean and std over 6-day and 24-day windows (approximating 6h/24h for daily data)
- **Cyclic encoding**: day-of-year (sin/cos), day-of-week (sin/cos)
- **Station identity**: one-hot encoded city column

Note: The Kaggle dataset is daily, not hourly, so lag periods are in days (t-1 = 1 day, not 1 hour).
For the backend serving hourly data, the same feature names are used but populated with hourly values.

In [ ]:
# Start with a clean copy
df = df_raw[['Date', 'City', 'AQI']].copy()
df = df.dropna(subset=['AQI'])  # can't train without target
df = df.sort_values(['City', 'Date']).reset_index(drop=True)

# Create station_id from city name (matches backend convention)
df['station_id'] = df['City'].str.lower().str.replace(r'\s+', '_', regex=True)

print(f'After dropping AQI nulls: {len(df)} rows, {df["City"].nunique()} cities')

In [ ]:
# ── Lag features ─────────────────────────────────────────────────────────
# For daily data: lag 1 = yesterday, lag 24 ≈ ~3.5 weeks, lag 168 ≈ ~24 weeks
# We'll use lag 1, 7 (week), and 30 (month) for daily data,
# but NAME them aqi_lag_1, aqi_lag_24, aqi_lag_168 to match the backend feature names
# (backend will use hourly data with actual 1h/24h/168h lags)

df['aqi_lag_1'] = df.groupby('station_id')['AQI'].shift(1)
df['aqi_lag_24'] = df.groupby('station_id')['AQI'].shift(7)    # 7-day lag (≈ weekly pattern)
df['aqi_lag_168'] = df.groupby('station_id')['AQI'].shift(30)  # 30-day lag (≈ monthly pattern)

# ── Rolling stats ────────────────────────────────────────────────────────
df['aqi_rolling_mean_6'] = df.groupby('station_id')['AQI'].transform(
    lambda x: x.rolling(window=6, min_periods=1).mean()
)
df['aqi_rolling_std_6'] = df.groupby('station_id')['AQI'].transform(
    lambda x: x.rolling(window=6, min_periods=1).std()
)
df['aqi_rolling_mean_24'] = df.groupby('station_id')['AQI'].transform(
    lambda x: x.rolling(window=24, min_periods=1).mean()
)
df['aqi_rolling_std_24'] = df.groupby('station_id')['AQI'].transform(
    lambda x: x.rolling(window=24, min_periods=1).std()
)

# Fill NaN std with 0 (happens when window has single value)
df['aqi_rolling_std_6'] = df['aqi_rolling_std_6'].fillna(0)
df['aqi_rolling_std_24'] = df['aqi_rolling_std_24'].fillna(0)

print('Lag and rolling features created.')

In [ ]:
# ── Cyclic time encoding ─────────────────────────────────────────────────
# hour_sin/hour_cos: using day-of-year as proxy for daily data
# (backend uses actual hour-of-day for hourly data)
day_of_year = df['Date'].dt.dayofyear
df['hour_sin'] = np.sin(2 * np.pi * day_of_year / 365)
df['hour_cos'] = np.cos(2 * np.pi * day_of_year / 365)

dow = df['Date'].dt.dayofweek
df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
df['dow_cos'] = np.cos(2 * np.pi * dow / 7)

print('Cyclic time features created.')

In [ ]:
# ── Weather features (placeholder) ───────────────────────────────────────
# The Kaggle dataset doesn't include weather — fill with defaults.
# The backend will use real OWM data for live predictions.
df['temp'] = 30.0
df['humidity'] = 60.0
df['wind_speed'] = 3.0
df['wind_deg'] = 180.0

print('Weather features added (defaults — no weather data in Kaggle CSV).')

In [ ]:
# ── Station one-hot encoding ─────────────────────────────────────────────
station_dummies = pd.get_dummies(df['station_id'], prefix='station', dtype=float)
df = pd.concat([df, station_dummies], axis=1)

print(f'Station one-hot columns: {len(station_dummies.columns)}')
print(f'Total columns: {len(df.columns)}')

In [ ]:
# ── Create target columns for different horizons ─────────────────────────
# target_24 = AQI 1 day ahead (daily data proxy)
# target_48 = AQI 2 days ahead
# target_72 = AQI 3 days ahead

df['target_24'] = df.groupby('station_id')['AQI'].shift(-1)
df['target_48'] = df.groupby('station_id')['AQI'].shift(-2)
df['target_72'] = df.groupby('station_id')['AQI'].shift(-3)

print('Target columns created (shift -1/-2/-3 days as proxy for 24h/48h/72h).')

In [ ]:
# ── Prepare training dataset with horizon as a feature ───────────────────
# Single-model approach: stack all horizons, add 'horizon' column

feature_cols = (
    ['aqi_lag_1', 'aqi_lag_24', 'aqi_lag_168']
    + ['aqi_rolling_mean_6', 'aqi_rolling_std_6', 'aqi_rolling_mean_24', 'aqi_rolling_std_24']
    + ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
    + ['temp', 'humidity', 'wind_speed', 'wind_deg']
    + ['horizon']
    + [c for c in df.columns if c.startswith('station_')]
)

# Stack horizons
rows = []
for h, target_col in [(24, 'target_24'), (48, 'target_48'), (72, 'target_72')]:
    subset = df.dropna(subset=[target_col] + ['aqi_lag_1']).copy()
    subset['horizon'] = h
    subset['target'] = subset[target_col]
    rows.append(subset)

df_model = pd.concat(rows, ignore_index=True)

# Ensure feature_cols exist (station_ columns from get_dummies)
feature_cols = [c for c in feature_cols if c in df_model.columns]

print(f'Model dataset: {len(df_model)} rows')
print(f'Feature columns ({len(feature_cols)}): {feature_cols[:10]}...')
df_model[feature_cols + ['target']].describe()

---
## 5. Train/Test Split

**Time-based split** — no random shuffle, this is time-series data (leakage matters).  
Hold out the most recent ~3 weeks as the test set.

In [ ]:
# Find the cutoff date: 21 days before the last date
max_date = df_model['Date'].max()
cutoff_date = max_date - pd.Timedelta(days=21)

train = df_model[df_model['Date'] <= cutoff_date].copy()
test = df_model[df_model['Date'] > cutoff_date].copy()

X_train = train[feature_cols]
y_train = train['target']
X_test = test[feature_cols]
y_test = test['target']

print(f'Train: {len(train)} rows ({train["Date"].min().date()} to {train["Date"].max().date()})')
print(f'Test:  {len(test)} rows ({test["Date"].min().date()} to {test["Date"].max().date()})')
print(f'Train/test ratio: {len(train)/len(df_model)*100:.1f}% / {len(test)/len(df_model)*100:.1f}%')

---
## 6. Persistence Baseline

The simplest forecast: predict AQI(t+h) = AQI(t).  
This is the benchmark the model must beat.

In [ ]:
# Persistence baseline: predict target = aqi_lag_1 (current AQI)
baseline_results = {}

for h in [24, 48, 72]:
    test_h = test[test['horizon'] == h].copy()
    if len(test_h) == 0:
        continue
    y_true = test_h['target']
    y_pred_baseline = test_h['aqi_lag_1']  # persistence: AQI(t+h) = AQI(t)
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_baseline))
    mae = mean_absolute_error(y_true, y_pred_baseline)
    
    baseline_results[h] = {'rmse': rmse, 'mae': mae}
    print(f'Horizon {h}h — Persistence Baseline: RMSE={rmse:.2f}, MAE={mae:.2f}')

print('\nThis is the bar the model needs to clear. ↑')

---
## 7. Model Training

### Approach: Single XGBoost model with `horizon` as a feature

**Why single model instead of 3 separate models?**
- **Pro**: One pkl file, simpler serialization, the model can learn shared patterns across horizons
- **Con**: May underperform on specific horizons vs dedicated models that specialize
- **Tradeoff**: For a hackathon prototype, simplicity wins. With more time, we'd train per-horizon models and ensemble them.

Early stopping on a held-out validation set (last 7 days of training data).

In [ ]:
# Split a validation set from the end of training data for early stopping
val_cutoff = train['Date'].max() - pd.Timedelta(days=7)
train_fit = train[train['Date'] <= val_cutoff]
train_val = train[train['Date'] > val_cutoff]

X_fit = train_fit[feature_cols]
y_fit = train_fit['target']
X_val = train_val[feature_cols]
y_val = train_val['target']

print(f'Fit set: {len(train_fit)} rows')
print(f'Val set: {len(train_val)} rows')

In [ ]:
# Train XGBoost regressor
model = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
)

model.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

print(f'\nBest iteration: {model.best_iteration}')
print(f'Best score (validation RMSE): {model.best_score:.4f}')

In [ ]:
# Feature importance (top 20)
importance = pd.Series(
    model.feature_importances_, index=feature_cols
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(20).plot.barh(ax=ax, color='#3498db')
ax.set_title('Top 20 Feature Importances', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 8. Evaluation

RMSE and MAE per horizon, compared against the persistence baseline.  
Scatter plots of predicted vs actual for visual inspection.

In [ ]:
# Predict on test set
y_pred = model.predict(X_test)
test = test.copy()
test['predicted'] = y_pred

# Metrics per horizon
model_results = {}
rows_table = []

for h in [24, 48, 72]:
    test_h = test[test['horizon'] == h]
    if len(test_h) == 0:
        continue
    y_true_h = test_h['target']
    y_pred_h = test_h['predicted']
    
    rmse = np.sqrt(mean_squared_error(y_true_h, y_pred_h))
    mae = mean_absolute_error(y_true_h, y_pred_h)
    
    model_results[h] = {'rmse': rmse, 'mae': mae}
    
    bl = baseline_results.get(h, {'rmse': 0, 'mae': 0})
    improvement = (bl['rmse'] - rmse) / bl['rmse'] * 100 if bl['rmse'] > 0 else 0
    
    rows_table.append({
        'Horizon': f'{h}h',
        'Baseline RMSE': f'{bl["rmse"]:.2f}',
        'Model RMSE': f'{rmse:.2f}',
        'Improvement': f'{improvement:.1f}%',
        'Baseline MAE': f'{bl["mae"]:.2f}',
        'Model MAE': f'{mae:.2f}',
    })

results_df = pd.DataFrame(rows_table)
print('\n=== Model vs Persistence Baseline ===')
print(results_df.to_string(index=False))

In [ ]:
# Predicted vs Actual scatter plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, h in zip(axes, [24, 48, 72]):
    test_h = test[test['horizon'] == h]
    if len(test_h) == 0:
        ax.set_title(f'{h}h — No data')
        continue
    
    ax.scatter(test_h['target'], test_h['predicted'], alpha=0.3, s=10, color='#2ecc71')
    
    # Perfect prediction line
    lims = [0, max(test_h['target'].max(), test_h['predicted'].max()) * 1.1]
    ax.plot(lims, lims, 'r--', alpha=0.5, label='Perfect prediction')
    
    rmse = model_results[h]['rmse']
    ax.set_title(f'{h}h Forecast — RMSE: {rmse:.1f}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Actual AQI')
    ax.set_ylabel('Predicted AQI')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Residual distributions per horizon
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, h in zip(axes, [24, 48, 72]):
    test_h = test[test['horizon'] == h]
    if len(test_h) == 0:
        continue
    residuals = test_h['target'] - test_h['predicted']
    ax.hist(residuals, bins=50, alpha=0.7, color='#9b59b6', edgecolor='white')
    ax.axvline(x=0, color='red', linestyle='--')
    ax.set_title(f'{h}h Residuals (mean={residuals.mean():.1f})', fontsize=13, fontweight='bold')
    ax.set_xlabel('Residual (Actual - Predicted)')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Time series comparison for a sample city (Delhi)
sample_city = 'delhi'  # station_id
test_sample = test[(test['station_id'] == sample_city) & (test['horizon'] == 24)].sort_values('Date')

if len(test_sample) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(test_sample['Date'], test_sample['target'], label='Actual', color='#2c3e50', linewidth=1.5)
    ax.plot(test_sample['Date'], test_sample['predicted'], label='Predicted (24h)', 
            color='#e74c3c', linewidth=1.5, linestyle='--')
    ax.fill_between(test_sample['Date'], 
                    test_sample['predicted'] - model_results[24]['rmse'],
                    test_sample['predicted'] + model_results[24]['rmse'],
                    alpha=0.15, color='#e74c3c', label='±1 RMSE')
    ax.set_title('Delhi — 24h AQI Forecast vs Actual (Test Set)', fontsize=14, fontweight='bold')
    ax.set_ylabel('AQI')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f'No test data for station_id={sample_city}')

---
## 9. Export

Serialize the model + metadata with joblib.  
The backend's `model_utils.py` loads this at startup.

In [ ]:
# Build the export artifact
export_dict = {
    'model': model,
    'feature_names': feature_cols,
    'training_rmse': {h: r['rmse'] for h, r in model_results.items()},
    'training_mae': {h: r['mae'] for h, r in model_results.items()},
    'baseline_rmse': {h: r['rmse'] for h, r in baseline_results.items()},
    'train_date_range': (str(train['Date'].min()), str(train['Date'].max())),
    'test_date_range': (str(test['Date'].min()), str(test['Date'].max())),
}

# Save locally
output_path = 'aqi_forecast.pkl'
joblib.dump(export_dict, output_path)
print(f'Model saved to: {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')

# Also save to Drive if mounted
if DRIVE_MOUNTED:
    drive_path = '/content/drive/MyDrive/aqi-intel/aqi_forecast.pkl'
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    joblib.dump(export_dict, drive_path)
    print(f'Also saved to Drive: {drive_path}')

In [ ]:
# Print the exact feature names — backend MUST match this order
print('=== FEATURE NAMES (copy to backend if needed) ===')
print(f'Total features: {len(feature_cols)}')
print()
for i, name in enumerate(feature_cols):
    print(f'  [{i:3d}] {name}')

In [ ]:
# Quick sanity check — reload and predict
loaded = joblib.load(output_path)
test_pred = loaded['model'].predict(X_test.head(5))
print('Sanity check predictions:', test_pred)
print('Feature names match:', loaded['feature_names'] == feature_cols)

---
## 10. Future Improvements

What would make this better with more time:

1. **Satellite NO₂ features** — Sentinel-5P tropospheric NO₂ column density as an additional predictor. Available via Google Earth Engine at ~daily resolution, ~5km spatial.

2. **Spatial interpolation** — Instead of treating each station independently, use inverse-distance weighting or kriging to interpolate AQI between stations. This enables truly *hyperlocal* predictions at any lat/lon, not just at monitor locations.

3. **Hourly data** — Switch from Kaggle daily CSV to the CPCB CAAQMS hourly data (available via the CCR dashboard). This unlocks real t-1h, t-24h lags and captures diurnal patterns the daily data misses.

4. **Weather forecast integration** — Instead of current weather, use NWP forecast weather (e.g., GFS or OpenWeatherMap 5-day forecast) as features. The model then predicts AQI at t+h given *forecasted* weather at t+h, not just current weather at t.

5. **Graph Neural Network (GNN)** — Model the station network as a graph, with edges weighted by distance. A GNN could learn spatial correlations (e.g., pollution from an industrial zone affecting downwind residential stations).

6. **Attention-based temporal model** — Replace XGBoost with a temporal attention model (e.g., Temporal Fusion Transformer) that can learn long-range dependencies and provide interpretable attention weights.

7. **Ensemble** — Blend XGBoost with a simple LSTM or 1D-CNN for diversity. Stacking typically improves RMSE by 5-10% on time series tasks.

8. **Confidence calibration** — Replace the naive ±RMSE band with conformal prediction intervals (e.g., MAPIE) for calibrated coverage guarantees.

9. **Source apportionment** — Add features for known emission events (stubble burning season, Diwali, construction activity) as binary/seasonal indicators.